<a href="https://colab.research.google.com/github/Johnbaby2002/Diabetic-Retinopathy/blob/main/Retinopathy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os
os.listdir('/content/drive')

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']

In [ ]:
import os
os.listdir('/content/drive/MyDrive')

['Netzwerk neu A2 KB_copy.pdf',
 'index.gdoc',
 'index-1.gdoc',
 'clideo.com',
 '4617a0ac4175116af4c755112bfec8c9be61d40a_copy.pdf',
 'this is a test (1).gsheet',
 'test n8n',
 'this is a test.gsheet',
 'n8n test.gdoc',
 'kpi_monthly_static-Grid view.csv',
 'performance dashboard.gsheet',
 'krankmeldung.pdf',
 'Untitled document.gdoc',
 'JohnNayathodan_CV (1).pdf',
 'JohnNayathodan_CV.pdf',
 'Fenonia(1667-).gdoc',
 'Colab Notebooks',
 'aptos2019-blindness-detection']

In [ ]:
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")
os.listdir(BASE_PATH)


['sample_submission.csv',
 'test.csv',
 'train.csv',
 'test_images',
 'train_images']

In [ ]:
import os
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")

for root, dirs, files in os.walk(BASE_PATH):
    png_count = len([f for f in files if f.endswith(".png")])
    if png_count > 0:
        print(root, "PNG files:", png_count)

/content/drive/MyDrive/aptos2019-blindness-detection/test_images PNG files: 1928
/content/drive/MyDrive/aptos2019-blindness-detection/train_images PNG files: 3665


In [ ]:
BASE_PATH = "/content/drive/MyDrive/aptos2019-blindness-detection"

import os

print("Train images:", len(os.listdir(BASE_PATH + "/train_images")))
print("Test images:", len(os.listdir(BASE_PATH + "/test_images")))

Train images: 3665
Test images: 1928


In [ ]:
import os

os.listdir('/content/drive/MyDrive')


['Netzwerk neu A2 KB_copy.pdf',
 'index.gdoc',
 'index-1.gdoc',
 'clideo.com',
 '4617a0ac4175116af4c755112bfec8c9be61d40a_copy.pdf',
 'this is a test (1).gsheet',
 'test n8n',
 'this is a test.gsheet',
 'n8n test.gdoc',
 'kpi_monthly_static-Grid view.csv',
 'performance dashboard.gsheet',
 'krankmeldung.pdf',
 'Untitled document.gdoc',
 'JohnNayathodan_CV (1).pdf',
 'JohnNayathodan_CV.pdf',
 'Fenonia(1667-).gdoc',
 'Colab Notebooks',
 'aptos2019-blindness-detection']

In [ ]:

import os

BASE_PATH = "/content/drive/MyDrive/aptos2019-blindness-detection"

print(os.listdir(BASE_PATH))

['sample_submission.csv', 'test.csv', 'train.csv', 'test_images', 'train_images']


In [ ]:
for root, dirs, files in os.walk(BASE_PATH):
    if "train_images" in dirs or "train.csv" in files:
        print(root)


/content/drive/MyDrive/aptos2019-blindness-detection


In [ ]:
for root, dirs, files in os.walk(BASE_PATH):
    png_count = len([f for f in files if f.endswith(".png")])
    if png_count > 100:
        print(root, png_count)

/content/drive/MyDrive/aptos2019-blindness-detection/test_images 1928
/content/drive/MyDrive/aptos2019-blindness-detection/train_images 3665


In [ ]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")
TRAIN_CSV = BASE_PATH / "train.csv"
TRAIN_IMAGES = BASE_PATH / "train_images"

train_df = pd.read_csv(TRAIN_CSV)

train_df["image_path"] = train_df["id_code"].apply(
    lambda x: TRAIN_IMAGES / f"{x}.png"
)

print("Dataset shape:", train_df.shape)
print("Missing images:", sum(~train_df["image_path"].apply(lambda p: p.exists())))
print(train_df["diagnosis"].value_counts().sort_index())

Dataset shape: (3662, 3)
Missing images: 0
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [ ]:
# =====================
# FULL TRAINING IMAGE CLASSIFICATION OVERVIEW
# Run this before the model training code.
# =====================
from IPython.display import display

DIAGNOSIS_LABELS = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR",
}

classification_df = train_df[["id_code", "diagnosis", "image_path"]].copy()
classification_df["diagnosis_label"] = classification_df["diagnosis"].map(DIAGNOSIS_LABELS)
classification_df = classification_df[
    ["id_code", "image_path", "diagnosis", "diagnosis_label"]
].sort_values(["diagnosis", "id_code"]).reset_index(drop=True)

summary_df = (
    classification_df.groupby(["diagnosis", "diagnosis_label"], as_index=False)
    .size()
    .rename(columns={"size": "image_count"})
)
summary_df["percentage"] = (
    summary_df["image_count"] / len(classification_df) * 100
).round(2)

print("Training image classification summary:")
display(summary_df)

print("Full training image classification table:")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)
display(classification_df)

classification_csv = BASE_PATH / "training_image_classification.csv"
classification_df.to_csv(classification_csv, index=False)
print(f"Saved full classification table to: {classification_csv}")


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# =====================
# SETTINGS
# =====================
BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")

BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 1e-4

USE_SMALL_SAMPLE = True
TRAIN_SAMPLE_SIZE = 500
VAL_SAMPLE_SIZE = 150

# =====================
# FIND FILES AUTOMATICALLY
# =====================
print("Searching dataset files...")

train_csv_paths = []
train_image_paths = []

for root, dirs, files in os.walk(BASE_PATH):
    root_path = Path(root)

    if "train.csv" in files:
        train_csv_paths.append(root_path / "train.csv")

    if root_path.name == "train_images":
        train_image_paths.append(root_path)

print("Found train.csv:", train_csv_paths)
print("Found train_images:", train_image_paths)

if len(train_csv_paths) == 0:
    raise FileNotFoundError("train.csv not found inside BASE_PATH.")

if len(train_image_paths) == 0:
    raise FileNotFoundError("train_images folder not found inside BASE_PATH.")

TRAIN_CSV = train_csv_paths[0]
TRAIN_IMAGES = train_image_paths[0]

print("\nUsing TRAIN_CSV:", TRAIN_CSV)
print("Using TRAIN_IMAGES:", TRAIN_IMAGES)

# =====================
# LOAD DATA
# =====================
train_df = pd.read_csv(TRAIN_CSV)

train_df["image_path"] = train_df["id_code"].apply(
    lambda x: TRAIN_IMAGES / f"{x}.png"
)

print("\nOriginal dataset shape:", train_df.shape)
print("\nOriginal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

missing = sum(~train_df["image_path"].apply(lambda p: p.exists()))
print("\nMissing images:", missing)

# Safety: remove missing image rows
if missing > 0:
    print("Filtering missing images...")
    train_df = train_df[train_df["image_path"].apply(lambda p: p.exists())].reset_index(drop=True)

print("\nFinal dataset shape:", train_df.shape)
print("\nFinal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

# =====================
# SPLIT
# =====================
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["diagnosis"],
    random_state=42
)

if USE_SMALL_SAMPLE:
    print("\nUsing small sample mode...")

    train_split = train_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, TRAIN_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

    val_split = val_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, VAL_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

print("\nTrain shape:", train_split.shape)
print("Val shape:", val_split.shape)

print("\nTrain distribution:")
print(train_split["diagnosis"].value_counts().sort_index())

print("\nVal distribution:")
print(val_split["diagnosis"].value_counts().sort_index())

# =====================
# TRANSFORMS
# =====================
weights = EfficientNet_B0_Weights.DEFAULT
train_transform = weights.transforms()
val_transform = weights.transforms()

# =====================
# DATASET
# =====================
class AptosDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = str(row["image_path"])
        label = int(row["diagnosis"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = AptosDataset(train_split, train_transform)
val_dataset = AptosDataset(val_split, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================
# MODEL
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not enabled. Go to Runtime > Change runtime type > GPU.")

model = efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# =====================
# TRAIN FUNCTION
# =====================
def train_one_epoch():
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(train_loader), correct / total

# =====================
# EVALUATION FUNCTION
# =====================
def evaluate():
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return total_loss / len(val_loader), correct / total, all_labels, all_preds

# =====================
# TRAINING LOOP
# =====================
best_qwk = -1

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch + 1}/{EPOCHS} =====")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc, y_true, y_pred = evaluate()

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    print("\nEpoch Summary")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")
    print(f"Val QWK   : {qwk:.4f}")

    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), "best_efficientnet_b0_aptos.pth")
        print("Saved best model.")

# =====================
# FINAL RESULTS
# =====================
print("\nBest QWK:", best_qwk)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Searching dataset files...
Found train.csv: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train.csv')]
Found train_images: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train_images')]

Using TRAIN_CSV: /content/drive/MyDrive/aptos2019-blindness-detection/train.csv
Using TRAIN_IMAGES: /content/drive/MyDrive/aptos2019-blindness-detection/train_images

Original dataset shape: (3662, 3)

Original class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Missing images: 0

Final dataset shape: (3662, 3)

Final class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Using small sample mode...

Train shape: (500, 3)
Val shape: (150, 3)

Train distribution:
diagnosis
0    100
1    100
2    100
3    100
4    100
Name: count, dtype: int64

Val distribution:
diagnosis
0    30
1    30
2    30
3    30
4    30
Name: count, dtype: int64

Device: cuda
GPU: Tesla T4

/tmp/ipykernel_1294/1434486665.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_split = train_split.groupby("diagnosis", group_keys=False).apply(
/tmp/ipykernel_1294/1434486665.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_split = val_split.groupby("diagnosis", group_keys=False).apply(


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 140MB/s]



===== EPOCH 1/3 =====
Batch 0/32 | Loss: 1.5903
Batch 10/32 | Loss: 1.5974
Batch 20/32 | Loss: 1.4127
Batch 30/32 | Loss: 1.3490

Epoch Summary
Train Loss: 1.4806
Train Acc : 0.3520
Val Loss  : 1.3898
Val Acc   : 0.4133
Val QWK   : 0.4124
Saved best model.

===== EPOCH 2/3 =====
Batch 0/32 | Loss: 1.4638
Batch 10/32 | Loss: 1.3296
Batch 20/32 | Loss: 1.0068
Batch 30/32 | Loss: 0.9365

Epoch Summary
Train Loss: 1.1934
Train Acc : 0.6100
Val Loss  : 1.1956
Val Acc   : 0.5333
Val QWK   : 0.5837
Saved best model.

===== EPOCH 3/3 =====
Batch 0/32 | Loss: 1.0970
Batch 10/32 | Loss: 1.1639
Batch 20/32 | Loss: 0.8794
Batch 30/32 | Loss: 0.8584

Epoch Summary
Train Loss: 0.9991
Train Acc : 0.7180
Val Loss  : 1.1117
Val Acc   : 0.6000
Val QWK   : 0.6847
Saved best model.

Best QWK: 0.6847457627118644

Confusion Matrix:
[[29  1  0  0  0]
 [ 4 19  3  2  2]
 [ 2  5 17  2  4]
 [ 1  2  6 15  6]
 [ 0  6  8  6 10]]

Classification Report:
              precision    recall  f1-score   support

       

In [ ]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# =====================
# SETTINGS
# =====================
BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")

BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 1e-4

USE_SMALL_SAMPLE = True
TRAIN_SAMPLE_SIZE = 500
VAL_SAMPLE_SIZE = 150

# =====================
# FIND FILES AUTOMATICALLY
# =====================
print("Searching dataset files...")

train_csv_paths = []
train_image_paths = []

for root, dirs, files in os.walk(BASE_PATH):
    root_path = Path(root)

    if "train.csv" in files:
        train_csv_paths.append(root_path / "train.csv")

    if root_path.name == "train_images":
        train_image_paths.append(root_path)

print("Found train.csv:", train_csv_paths)
print("Found train_images:", train_image_paths)

if len(train_csv_paths) == 0:
    raise FileNotFoundError("train.csv not found inside BASE_PATH.")

if len(train_image_paths) == 0:
    raise FileNotFoundError("train_images folder not found inside BASE_PATH.")

TRAIN_CSV = train_csv_paths[0]
TRAIN_IMAGES = train_image_paths[0]

print("\nUsing TRAIN_CSV:", TRAIN_CSV)
print("Using TRAIN_IMAGES:", TRAIN_IMAGES)

# =====================
# LOAD DATA
# =====================
train_df = pd.read_csv(TRAIN_CSV)

train_df["image_path"] = train_df["id_code"].apply(
    lambda x: TRAIN_IMAGES / f"{x}.png"
)

print("\nOriginal dataset shape:", train_df.shape)
print("\nOriginal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

missing = sum(~train_df["image_path"].apply(lambda p: p.exists()))
print("\nMissing images:", missing)

# Safety: remove missing image rows
if missing > 0:
    print("Filtering missing images...")
    train_df = train_df[train_df["image_path"].apply(lambda p: p.exists())].reset_index(drop=True)

print("\nFinal dataset shape:", train_df.shape)
print("\nFinal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

# =====================
# SPLIT
# =====================
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["diagnosis"],
    random_state=42
)

if USE_SMALL_SAMPLE:
    print("\nUsing small sample mode...")

    train_split = train_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, TRAIN_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

    val_split = val_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, VAL_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

print("\nTrain shape:", train_split.shape)
print("Val shape:", val_split.shape)

print("\nTrain distribution:")
print(train_split["diagnosis"].value_counts().sort_index())

print("\nVal distribution:")
print(val_split["diagnosis"].value_counts().sort_index())

# =====================
# TRANSFORMS
# =====================
weights = EfficientNet_B0_Weights.DEFAULT
train_transform = weights.transforms()
val_transform = weights.transforms()

# =====================
# DATASET
# =====================
class AptosDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = str(row["image_path"])
        label = int(row["diagnosis"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = AptosDataset(train_split, train_transform)
val_dataset = AptosDataset(val_split, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================
# MODEL
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not enabled. Go to Runtime > Change runtime type > GPU.")

model = efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# =====================
# TRAIN FUNCTION
# =====================
def train_one_epoch():
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(train_loader), correct / total

# =====================
# EVALUATION FUNCTION
# =====================
def evaluate():
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return total_loss / len(val_loader), correct / total, all_labels, all_preds

# =====================
# TRAINING LOOP
# =====================
best_qwk = -1

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch + 1}/{EPOCHS} =====")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc, y_true, y_pred = evaluate()

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    print("\nEpoch Summary")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")
    print(f"Val QWK   : {qwk:.4f}")

    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), "best_efficientnet_b0_aptos.pth")
        print("Saved best model.")

# =====================
# FINAL RESULTS
# =====================
print("\nBest QWK:", best_qwk)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Searching dataset files...
Found train.csv: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train.csv')]
Found train_images: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train_images')]

Using TRAIN_CSV: /content/drive/MyDrive/aptos2019-blindness-detection/train.csv
Using TRAIN_IMAGES: /content/drive/MyDrive/aptos2019-blindness-detection/train_images

Original dataset shape: (3662, 3)

Original class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Missing images: 0

Final dataset shape: (3662, 3)

Final class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Using small sample mode...

Train shape: (500, 3)
Val shape: (150, 3)

Train distribution:
diagnosis
0    100
1    100
2    100
3    100
4    100
Name: count, dtype: int64

Val distribution:
diagnosis
0    30
1    30
2    30
3    30
4    30
Name: count, dtype: int64

Device: cuda
GPU: Tesla T4

/tmp/ipykernel_1294/1434486665.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_split = train_split.groupby("diagnosis", group_keys=False).apply(
/tmp/ipykernel_1294/1434486665.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_split = val_split.groupby("diagnosis", group_keys=False).apply(



===== EPOCH 1/3 =====
Batch 0/32 | Loss: 1.5555
Batch 10/32 | Loss: 1.6462
Batch 20/32 | Loss: 1.3558
Batch 30/32 | Loss: 1.2829

Epoch Summary
Train Loss: 1.4799
Train Acc : 0.3520
Val Loss  : 1.3920
Val Acc   : 0.3933
Val QWK   : 0.5024
Saved best model.

===== EPOCH 2/3 =====
Batch 0/32 | Loss: 1.4836
Batch 10/32 | Loss: 1.2134
Batch 20/32 | Loss: 1.0779
Batch 30/32 | Loss: 1.1680

Epoch Summary
Train Loss: 1.1942
Train Acc : 0.6120
Val Loss  : 1.2245
Val Acc   : 0.4867
Val QWK   : 0.5767
Saved best model.

===== EPOCH 3/3 =====
Batch 0/32 | Loss: 1.3481
Batch 10/32 | Loss: 0.7937
Batch 20/32 | Loss: 0.8731
Batch 30/32 | Loss: 0.9476

Epoch Summary
Train Loss: 1.0025
Train Acc : 0.7020
Val Loss  : 1.1339
Val Acc   : 0.5533
Val QWK   : 0.6866
Saved best model.

Best QWK: 0.6866141732283464

Confusion Matrix:
[[30  0  0  0  0]
 [ 6 17  0  3  4]
 [ 3 11  7  7  2]
 [ 2  0  2 18  8]
 [ 0  6  4  9 11]]

Classification Report:
              precision    recall  f1-score   support

       

In [ ]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# =====================
# SETTINGS
# =====================
BASE_PATH = Path("/content/drive/MyDrive/aptos2019-blindness-detection")

BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 1e-4

USE_SMALL_SAMPLE = True
TRAIN_SAMPLE_SIZE = 500
VAL_SAMPLE_SIZE = 150

# =====================
# FIND FILES AUTOMATICALLY
# =====================
print("Searching dataset files...")

train_csv_paths = []
train_image_paths = []

for root, dirs, files in os.walk(BASE_PATH):
    root_path = Path(root)

    if "train.csv" in files:
        train_csv_paths.append(root_path / "train.csv")

    if root_path.name == "train_images":
        train_image_paths.append(root_path)

print("Found train.csv:", train_csv_paths)
print("Found train_images:", train_image_paths)

if len(train_csv_paths) == 0:
    raise FileNotFoundError("train.csv not found inside BASE_PATH.")

if len(train_image_paths) == 0:
    raise FileNotFoundError("train_images folder not found inside BASE_PATH.")

TRAIN_CSV = train_csv_paths[0]
TRAIN_IMAGES = train_image_paths[0]

print("\nUsing TRAIN_CSV:", TRAIN_CSV)
print("Using TRAIN_IMAGES:", TRAIN_IMAGES)

# =====================
# LOAD DATA
# =====================
train_df = pd.read_csv(TRAIN_CSV)

train_df["image_path"] = train_df["id_code"].apply(
    lambda x: TRAIN_IMAGES / f"{x}.png"
)

print("\nOriginal dataset shape:", train_df.shape)
print("\nOriginal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

missing = sum(~train_df["image_path"].apply(lambda p: p.exists()))
print("\nMissing images:", missing)

# Safety: remove missing image rows
if missing > 0:
    print("Filtering missing images...")
    train_df = train_df[train_df["image_path"].apply(lambda p: p.exists())].reset_index(drop=True)

print("\nFinal dataset shape:", train_df.shape)
print("\nFinal class distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

# =====================
# SPLIT
# =====================
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["diagnosis"],
    random_state=42
)

if USE_SMALL_SAMPLE:
    print("\nUsing small sample mode...")

    train_split = train_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, TRAIN_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

    val_split = val_split.groupby("diagnosis", group_keys=False).apply(
        lambda x: x.sample(
            n=min(len(x), max(1, VAL_SAMPLE_SIZE // train_df["diagnosis"].nunique())),
            random_state=42
        )
    ).reset_index(drop=True)

print("\nTrain shape:", train_split.shape)
print("Val shape:", val_split.shape)

print("\nTrain distribution:")
print(train_split["diagnosis"].value_counts().sort_index())

print("\nVal distribution:")
print(val_split["diagnosis"].value_counts().sort_index())

# =====================
# TRANSFORMS
# =====================
weights = EfficientNet_B0_Weights.DEFAULT
train_transform = weights.transforms()
val_transform = weights.transforms()

# =====================
# DATASET
# =====================
class AptosDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = str(row["image_path"])
        label = int(row["diagnosis"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = AptosDataset(train_split, train_transform)
val_dataset = AptosDataset(val_split, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================
# MODEL
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not enabled. Go to Runtime > Change runtime type > GPU.")

model = efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# =====================
# TRAIN FUNCTION
# =====================
def train_one_epoch():
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(train_loader), correct / total

# =====================
# EVALUATION FUNCTION
# =====================
def evaluate():
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return total_loss / len(val_loader), correct / total, all_labels, all_preds

# =====================
# TRAINING LOOP
# =====================
best_qwk = -1

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch + 1}/{EPOCHS} =====")

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc, y_true, y_pred = evaluate()

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    print("\nEpoch Summary")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")
    print(f"Val QWK   : {qwk:.4f}")

    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), "best_efficientnet_b0_aptos.pth")
        print("Saved best model.")

# =====================
# FINAL RESULTS
# =====================
print("\nBest QWK:", best_qwk)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Searching dataset files...
Found train.csv: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train.csv')]
Found train_images: [PosixPath('/content/drive/MyDrive/aptos2019-blindness-detection/train_images')]

Using TRAIN_CSV: /content/drive/MyDrive/aptos2019-blindness-detection/train.csv
Using TRAIN_IMAGES: /content/drive/MyDrive/aptos2019-blindness-detection/train_images

Original dataset shape: (3662, 3)

Original class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Missing images: 0

Final dataset shape: (3662, 3)

Final class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Using small sample mode...

Train shape: (500, 3)
Val shape: (150, 3)

Train distribution:
diagnosis
0    100
1    100
2    100
3    100
4    100
Name: count, dtype: int64

Val distribution:
diagnosis
0    30
1    30
2    30
3    30
4    30
Name: count, dtype: int64

Device: cuda
GPU: Tesla T4

/tmp/ipykernel_3546/1434486665.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_split = train_split.groupby("diagnosis", group_keys=False).apply(
/tmp/ipykernel_3546/1434486665.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_split = val_split.groupby("diagnosis", group_keys=False).apply(


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 123MB/s]



===== EPOCH 1/3 =====
Batch 0/32 | Loss: 1.5895
Batch 10/32 | Loss: 1.5315
Batch 20/32 | Loss: 1.5923
Batch 30/32 | Loss: 1.3042

Epoch Summary
Train Loss: 1.4877
Train Acc : 0.3540
Val Loss  : 1.3608
Val Acc   : 0.4667
Val QWK   : 0.5846
Saved best model.

===== EPOCH 2/3 =====
Batch 0/32 | Loss: 1.4099
Batch 10/32 | Loss: 1.2862
Batch 20/32 | Loss: 1.0125
Batch 30/32 | Loss: 1.0688

Epoch Summary
Train Loss: 1.2056
Train Acc : 0.6260
Val Loss  : 1.2125
Val Acc   : 0.5067
Val QWK   : 0.6147
Saved best model.

===== EPOCH 3/3 =====
Batch 0/32 | Loss: 1.1763
Batch 10/32 | Loss: 1.0159
Batch 20/32 | Loss: 1.0722
Batch 30/32 | Loss: 0.7190

Epoch Summary
Train Loss: 0.9683
Train Acc : 0.7100
Val Loss  : 1.1424
Val Acc   : 0.5600
Val QWK   : 0.7067
Saved best model.

Best QWK: 0.7066666666666667

Confusion Matrix:
[[30  0  0  0  0]
 [ 5 17  3  2  3]
 [ 3 10  8  7  2]
 [ 1  0  7 19  3]
 [ 0  6  3 11 10]]

Classification Report:
              precision    recall  f1-score   support

       